---
# Logsiz va Log + Feature Engineering

## Rawdataset yuklash

In [86]:
import pandas as pd
df=pd.read_csv('Job_Placement_Data.csv')

In [87]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               215 non-null    str    
 1   ssc_percentage       215 non-null    float64
 2   ssc_board            215 non-null    str    
 3   hsc_percentage       215 non-null    float64
 4   hsc_board            215 non-null    str    
 5   hsc_subject          215 non-null    str    
 6   degree_percentage    215 non-null    float64
 7   undergrad_degree     215 non-null    str    
 8   work_experience      215 non-null    str    
 9   emp_test_percentage  215 non-null    float64
 10  specialisation       215 non-null    str    
 11  mba_percent          215 non-null    float64
 12  status               215 non-null    str    
dtypes: float64(5), str(8)
memory usage: 22.0 KB


In [88]:
df.head()

,gender,ssc_percentage,ssc_board,hsc_percentage,hsc_board,hsc_subject,degree_percentage,undergrad_degree,work_experience,emp_test_percentage,specialisation,mba_percent,status
0,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed
1,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed
2,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed
3,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed
4,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed


---
## Preprocessing with Class

In [89]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [90]:
class DataPreprocessing:
    def __init__(self,df):
        self.df=df
    def tozala(self):
        for col in self.df.columns:
            if self.df[col].isnull().any():
                if self.df[col].dtype=='str':
                    self.df[col].fillna(self.df[col].mode()[0],inplace=True)
                else:
                    self.df[col].fillna(self.df[col].mean(),inplace=True)
        return self
    def encodla(self):
        encoder = LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype=='str':
                if self.df[col].nunique()<=5:
                    dummies=pd.get_dummies(self.df[col],prefix=col,dtype=int)
                    self.df=pd.concat([self.df.drop(columns=[col]),dummies],axis=1)
                else:
                    self.df[col]=encoder.fit_transform(self.df[col])
        return self
    def scale_qil(self):
        scaler=MinMaxScaler()
        for col in self.df.columns:
            num_col=self.df.select_dtypes(include=['float64','int64']).columns.drop('emp_test_percentage')
            self.df[num_col]=scaler.fit_transform(self.df[num_col])
        return self

In [91]:
dp=DataPreprocessing(df)
dp.tozala().encodla().scale_qil()
df=dp.df

In [92]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ssc_percentage              215 non-null    float64
 1   hsc_percentage              215 non-null    float64
 2   degree_percentage           215 non-null    float64
 3   emp_test_percentage         215 non-null    float64
 4   mba_percent                 215 non-null    float64
 5   gender_F                    215 non-null    float64
 6   gender_M                    215 non-null    float64
 7   ssc_board_Central           215 non-null    float64
 8   ssc_board_Others            215 non-null    float64
 9   hsc_board_Central           215 non-null    float64
 10  hsc_board_Others            215 non-null    float64
 11  hsc_subject_Arts            215 non-null    float64
 12  hsc_subject_Commerce        215 non-null    float64
 13  hsc_subject_Science         215 non-null    fl

---
# train-test-split

In [93]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['emp_test_percentage'])
y = df['emp_test_percentage']

In [94]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [95]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [96]:
# Linear Regression
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
print("r2: ", r2_lr)
print("mse: ", mse_lr)
print("mae: ", mae_lr)

r2:  -0.12711730604073002
mse:  179.57879467001257
mae:  11.346517996706687


In [97]:
# Ridge Regression (L1)
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
print("r2: ", r2_ridge)
print("mse: ", mse_ridge)
print("mae: ", mae_ridge)

r2:  -0.12262776671470954
mse:  178.86349551129197
mae:  11.327141069339154


In [98]:
# Lasso Regression (L2)
from sklearn.linear_model import Lasso
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
print("r2: ", r2_lasso)
print("mse: ", mse_lasso)
print("mae: ", mae_lasso)

r2:  -0.10085414596594378
mse:  175.39439735379813
mae:  11.341111750567842


In [99]:
# Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor
dtr = DecisionTreeRegressor()
dtr.fit(X_train, y_train)
y_pred_dtr = dtr.predict(X_test)
r2_dtr = r2_score(y_test, y_pred_dtr)
mse_dtr = mean_squared_error(y_test, y_pred_dtr)
mae_dtr = mean_absolute_error(y_test, y_pred_dtr)
print("r2: ", r2_dtr)
print("mse: ", mse_dtr)
print("mae: ", mae_dtr)

r2:  -0.8031986118608301
mse:  287.2959465116279
mae:  14.19093023255814


In [100]:
# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(n_estimators=100, random_state=42)
rfr.fit(X_train, y_train)
y_pred_rfr = rfr.predict(X_test)
r2_rfr = r2_score(y_test, y_pred_rfr)
mse_rfr = mean_squared_error(y_test, y_pred_rfr)
mae_rfr = mean_absolute_error(y_test, y_pred_rfr)
print("r2: ", r2_rfr)
print("mse: ", mse_rfr)
print("mae: ", mae_rfr)

r2:  -0.04771284394178221
mse:  166.92762028139535
mae:  11.267102325581394


In [101]:
# SVR
from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=1.0)
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)
r2_svr = r2_score(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
print("r2: ", r2_svr)
print("mse: ", mse_svr)
print("mae: ", mae_svr)

r2:  -0.030183534030678638
mse:  164.13475007315517
mae:  11.107523993670826


## Tabulate

In [102]:
from tabulate import tabulate

rows_reg = [
    ['Linear Regression', round(r2_lr, 4), round(mae_lr, 2), round(mse_lr, 2)],
    ['Ridge',             round(r2_ridge, 4), round(mae_ridge, 2), round(mse_ridge, 2)],
    ['Lasso',             round(r2_lasso, 4), round(mae_lasso, 2), round(mse_lasso, 2)],
    ['Decision Tree Regressor',     round(r2_dtr, 4), round(mae_dtr, 2), round(mse_dtr, 2)],
    ['Random Forest Regressor',     round(r2_rfr, 4), round(mae_rfr, 2), round(mse_rfr, 2)],
    ['SVR',               round(r2_svr, 4), round(mae_svr, 2), round(mse_svr, 2)]
]

print(tabulate(rows_reg, headers=['Model', 'R2', 'MAE', 'MSE'], tablefmt='fancy_grid'))

╒═════════════════════════╤═════════╤═══════╤════════╕
│ Model                   │      R2 │   MAE │    MSE │
╞═════════════════════════╪═════════╪═══════╪════════╡
│ Linear Regression       │ -0.1271 │ 11.35 │ 179.58 │
├─────────────────────────┼─────────┼───────┼────────┤
│ Ridge                   │ -0.1226 │ 11.33 │ 178.86 │
├─────────────────────────┼─────────┼───────┼────────┤
│ Lasso                   │ -0.1009 │ 11.34 │ 175.39 │
├─────────────────────────┼─────────┼───────┼────────┤
│ Decision Tree Regressor │ -0.8032 │ 14.19 │ 287.3  │
├─────────────────────────┼─────────┼───────┼────────┤
│ Random Forest Regressor │ -0.0477 │ 11.27 │ 166.93 │
├─────────────────────────┼─────────┼───────┼────────┤
│ SVR                     │ -0.0302 │ 11.11 │ 164.13 │
╘═════════════════════════╧═════════╧═══════╧════════╛


---
## with log + FE

In [103]:
import pandas as pd
df=pd.read_csv('Job_Placement_Data.csv')

In [104]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               215 non-null    str    
 1   ssc_percentage       215 non-null    float64
 2   ssc_board            215 non-null    str    
 3   hsc_percentage       215 non-null    float64
 4   hsc_board            215 non-null    str    
 5   hsc_subject          215 non-null    str    
 6   degree_percentage    215 non-null    float64
 7   undergrad_degree     215 non-null    str    
 8   work_experience      215 non-null    str    
 9   emp_test_percentage  215 non-null    float64
 10  specialisation       215 non-null    str    
 11  mba_percent          215 non-null    float64
 12  status               215 non-null    str    
dtypes: float64(5), str(8)
memory usage: 22.0 KB


# Preprocessing

In [105]:
df.isnull().sum()

gender                 0
ssc_percentage         0
ssc_board              0
hsc_percentage         0
hsc_board              0
hsc_subject            0
degree_percentage      0
undergrad_degree       0
work_experience        0
emp_test_percentage    0
specialisation         0
mba_percent            0
status                 0
dtype: int64

In [106]:
# missing
def tozala(df):
    for col in df.columns:        
        if df[col].isnull().any():
            if df[col].dtype=='str':
                df[col]=df[col].fillna(df[col].mode()[0])
            else:
                df[col]=df[col].fillna(df[col].mean())
    return df
df=tozala(df)

In [107]:
# encoding
from sklearn.preprocessing import LabelEncoder
def encodla(df):
    encoder=LabelEncoder()
    for col in df.columns:
        if df[col].dtype=='str':
            if df[col].nunique()<=5:
                dummies=pd.get_dummies(df[col],prefix=col,dtype=int)
                df=pd.concat([df.drop(columns=[col]),dummies],axis=1)
            else:
                df[col]=encoder.fit_transform(df[col])
    return df
df=encodla(df)

In [108]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ssc_percentage              215 non-null    float64
 1   hsc_percentage              215 non-null    float64
 2   degree_percentage           215 non-null    float64
 3   emp_test_percentage         215 non-null    float64
 4   mba_percent                 215 non-null    float64
 5   gender_F                    215 non-null    int64  
 6   gender_M                    215 non-null    int64  
 7   ssc_board_Central           215 non-null    int64  
 8   ssc_board_Others            215 non-null    int64  
 9   hsc_board_Central           215 non-null    int64  
 10  hsc_board_Others            215 non-null    int64  
 11  hsc_subject_Arts            215 non-null    int64  
 12  hsc_subject_Commerce        215 non-null    int64  
 13  hsc_subject_Science         215 non-null    in

## Log Transformation

In [109]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
num_cols

Index(['ssc_percentage', 'hsc_percentage', 'degree_percentage',
       'emp_test_percentage', 'mba_percent', 'gender_F', 'gender_M',
       'ssc_board_Central', 'ssc_board_Others', 'hsc_board_Central',
       'hsc_board_Others', 'hsc_subject_Arts', 'hsc_subject_Commerce',
       'hsc_subject_Science', 'undergrad_degree_Comm&Mgmt',
       'undergrad_degree_Others', 'undergrad_degree_Sci&Tech',
       'work_experience_No', 'work_experience_Yes', 'specialisation_Mkt&Fin',
       'specialisation_Mkt&HR', 'status_Not Placed', 'status_Placed'],
      dtype='str')

In [110]:
skewness = df[num_cols].skew()

In [111]:
import numpy as np
log_transformation = skewness[(skewness>=0.5)].index.tolist()
log_transformation

['gender_F',
 'hsc_subject_Arts',
 'undergrad_degree_Others',
 'undergrad_degree_Sci&Tech',
 'work_experience_Yes',
 'status_Not Placed']

In [112]:
# Logni qo'llash
for col in log_transformation:
    df[col+"_log"]=np.log1p(df[col])

In [113]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ssc_percentage                 215 non-null    float64
 1   hsc_percentage                 215 non-null    float64
 2   degree_percentage              215 non-null    float64
 3   emp_test_percentage            215 non-null    float64
 4   mba_percent                    215 non-null    float64
 5   gender_F                       215 non-null    int64  
 6   gender_M                       215 non-null    int64  
 7   ssc_board_Central              215 non-null    int64  
 8   ssc_board_Others               215 non-null    int64  
 9   hsc_board_Central              215 non-null    int64  
 10  hsc_board_Others               215 non-null    int64  
 11  hsc_subject_Arts               215 non-null    int64  
 12  hsc_subject_Commerce           215 non-null    int64  
 13  h

---
## Feature Engineering

In [114]:
# Bu talabaning umumiy o'qish darajasini ko'rsatadi
df['avg_academic'] = (df['ssc_percentage'] + df['hsc_percentage'] + df['degree_percentage']) / 3

In [115]:
# MBA va test nisbatini olish
df['mba_test_ratio'] = (df['mba_percent'] / (df['emp_test_percentage'] + 1))

In [116]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ssc_percentage                 215 non-null    float64
 1   hsc_percentage                 215 non-null    float64
 2   degree_percentage              215 non-null    float64
 3   emp_test_percentage            215 non-null    float64
 4   mba_percent                    215 non-null    float64
 5   gender_F                       215 non-null    int64  
 6   gender_M                       215 non-null    int64  
 7   ssc_board_Central              215 non-null    int64  
 8   ssc_board_Others               215 non-null    int64  
 9   hsc_board_Central              215 non-null    int64  
 10  hsc_board_Others               215 non-null    int64  
 11  hsc_subject_Arts               215 non-null    int64  
 12  hsc_subject_Commerce           215 non-null    int64  
 13  h

In [117]:
from sklearn.preprocessing import MinMaxScaler
def scale_qil(df):
    mms = MinMaxScaler()
    for col in df.columns:
        num_col = df.select_dtypes(include=['int64','float64']).columns.drop('emp_test_percentage')
        df[num_col] = mms.fit_transform(df[num_col])
    return df
df=scale_qil(df)

## train-test-split

In [118]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['emp_test_percentage'])
y = df['emp_test_percentage']

In [119]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [120]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [121]:
# Linear Regression
from sklearn.linear_model import LinearRegression
lr2 = LinearRegression()
lr2.fit(X_train, y_train)
y_pred_lr2 = lr2.predict(X_test)
r2_lr2 = r2_score(y_test, y_pred_lr2)
mse_lr2 = mean_squared_error(y_test, y_pred_lr2)
mae_lr2 = mean_absolute_error(y_test, y_pred_lr2)
print("r2: ", r2_lr2)
print("mse: ", mse_lr2)
print("mae: ", mae_lr2)

r2:  0.9799833411291541
mse:  3.1891689126610134
mae:  1.5953080982875543


In [122]:
# Ridge Regression (L1)
from sklearn.linear_model import Ridge
ridge2 = Ridge(alpha=0.1)
ridge2.fit(X_train, y_train)
y_pred_ridge2 = ridge2.predict(X_test)
r2_ridge2 = r2_score(y_test, y_pred_ridge2)
mse_ridge2 = mean_squared_error(y_test, y_pred_ridge2)
mae_ridge2 = mean_absolute_error(y_test, y_pred_ridge2)
print("r2: ", r2_ridge2)
print("mse: ", mse_ridge2)
print("mae: ", mae_ridge2)

r2:  0.9780437067511095
mse:  3.4982025880761345
mae:  1.6794947497622146


In [123]:
# Lasso Regression (L2)
from sklearn.linear_model import Lasso
lasso2 = Lasso(alpha=0.1)
lasso2.fit(X_train, y_train)
y_pred_lasso2 = lasso2.predict(X_test)
r2_lasso2 = r2_score(y_test, y_pred_lasso2)
mse_lasso2 = mean_squared_error(y_test, y_pred_lasso2)
mae_lasso2 = mean_absolute_error(y_test, y_pred_lasso2)
print("r2: ", r2_lasso2)
print("mse: ", mse_lasso2)
print("mae: ", mae_lasso2)

r2:  0.9741926494081825
mse:  4.111775134732558
mae:  1.7781154701204474


In [124]:
# Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor
dtr2 = DecisionTreeRegressor(random_state=42)
dtr2.fit(X_train, y_train)
y_pred_dtr2 = dtr2.predict(X_test)
r2_dtr2 = r2_score(y_test, y_pred_dtr2)
mse_dtr2 = mean_squared_error(y_test, y_pred_dtr2)
mae_dtr2 = mean_absolute_error(y_test, y_pred_dtr2)
print("r2: ", r2_dtr2)
print("mse: ", mse_dtr2)
print("mae: ", mae_dtr2)

r2:  0.9358796565568983
mse:  10.216020930232562
mae:  2.5011627906976748


In [125]:
# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
rfr2 = RandomForestRegressor(n_estimators=100, random_state=42)
rfr2.fit(X_train, y_train)
y_pred_rfr2 = rfr2.predict(X_test)
r2_rfr2 = r2_score(y_test, y_pred_rfr2)
mse_rfr2 = mean_squared_error(y_test, y_pred_rfr2)
mae_rfr2 = mean_absolute_error(y_test, y_pred_rfr2)
print("r2: ", r2_rfr2)
print("mse: ", mse_rfr2)
print("mae: ", mae_rfr2)

r2:  0.9545951431626717
mse:  7.234162246744214
mae:  2.033876744186046


In [126]:
# SVR
from sklearn.svm import SVR
svr2 = SVR(kernel='linear', C=10.0)
svr2.fit(X_train, y_train)
y_pred_svr2 = svr2.predict(X_test)
r2_svr2 = r2_score(y_test, y_pred_svr2)
mse_svr2 = mean_squared_error(y_test, y_pred_svr2)
mae_svr2 = mean_absolute_error(y_test, y_pred_svr2)
print("r2: ", r2_svr2)
print("mse: ", mse_svr2)
print("mae: ", mae_svr2)

r2:  0.9700658518818377
mse:  4.769280189136723
mae:  1.776152498069373


# Tabulate

In [130]:
from tabulate import tabulate

rows_reg = [
    ['Linear Regression', round(r2_lr, 4), round(r2_lr2, 4), round(mae_lr, 2), round(mae_lr2), round(mse_lr, 2), round(mse_lr2, 2)],
    ['Ridge',             round(r2_ridge, 4), round(r2_ridge2, 4), round(mae_ridge, 2), round(mae_ridge2, 2), round(mse_ridge, 2), round(mse_ridge2, 2)],
    ['Lasso',             round(r2_lasso, 4), round(r2_lasso2, 4), round(mae_lasso, 2), round(mae_lasso2, 2), round(mse_lasso, 2), round(mse_lasso2)],
    ['Decision Tree Regressor',     round(r2_dtr, 4), round(r2_dtr2, 4), round(mae_dtr, 2), round(mae_dtr2, 2), round(mse_dtr, 2), round(mse_dtr2, 2)],
    ['Random Forest Regressor',     round(r2_rfr, 4), round(r2_rfr2, 4), round(mae_rfr, 2), round(mae_rfr2, 2), round(mse_rfr, 2), round(mse_rfr2, 2)],
    ['SVR',               round(r2_svr, 4), round(r2_svr2, 4), round(mae_svr, 2), round(mae_svr2, 2), round(mse_svr, 2), round(mse_svr2, 2)]
]

print(tabulate(rows_reg, headers=['Model', 'R2', 'R2+log+FE', 'MAE', 'MAE+log+FE', 'MSE', 'MSE+log+FE'], tablefmt='fancy_grid'))

╒═════════════════════════╤═════════╤═════════════╤═══════╤══════════════╤════════╤══════════════╕
│ Model                   │      R2 │   R2+log+FE │   MAE │   MAE+log+FE │    MSE │   MSE+log+FE │
╞═════════════════════════╪═════════╪═════════════╪═══════╪══════════════╪════════╪══════════════╡
│ Linear Regression       │ -0.1271 │      0.98   │ 11.35 │         2    │ 179.58 │         3.19 │
├─────────────────────────┼─────────┼─────────────┼───────┼──────────────┼────────┼──────────────┤
│ Ridge                   │ -0.1226 │      0.978  │ 11.33 │         1.68 │ 178.86 │         3.5  │
├─────────────────────────┼─────────┼─────────────┼───────┼──────────────┼────────┼──────────────┤
│ Lasso                   │ -0.1009 │      0.9742 │ 11.34 │         1.78 │ 175.39 │         4    │
├─────────────────────────┼─────────┼─────────────┼───────┼──────────────┼────────┼──────────────┤
│ Decision Tree Regressor │ -0.8032 │      0.9359 │ 14.19 │         2.5  │ 287.3  │        10.22 │
├─────────

---
## Xulosa
***---> Men bundan xulosa oldimki Feature Engineering (FE) - odatda errorlar bilan ishlab errorlarni kamaytirishga katta hissa qo'shadi.***  
***---> Log transformation esa ko'proq r2_score va accuracy ni oshirishga yordam beradi.***